# The Augmented Dickey-Fuller Test

Wiki reference for [the ADF test](https://ml-viz-ruby.vercel.app/wiki/augmented-dickey-fuller).

**The idea in one sentence.** The ADF test checks for a **unit root** (non-stationarity) by
regressing $\Delta y_t$ on $y_{t-1}$ and testing whether the coefficient is zero — a very
negative $t$-statistic rejects the unit root — but you must choose the right **deterministic
term** ('c' vs 'ct') and confirm with **KPSS**, whose null is the opposite.

We build the Dickey-Fuller $t$-statistic from scratch, **validate it on a random walk vs a
stationary series and confirm ADF/KPSS agreement**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1. Stationarity is a statement about a root

For an AR(1) process $y_t = \phi\,y_{t-1} + \varepsilon_t$, the characteristic
polynomial is $1 - \phi z = 0$ with root $z = 1/\phi$.

- $|\phi| < 1$ → root **outside** the unit circle → **stationary** (mean-reverting)
- $\phi = 1$ → root **on** the circle → **unit root** → random walk (non-stationary)

Watch the same noise produce wildly different behaviour as $\phi$ crosses 1.

In [ ]:
T = 200
eps = rng.normal(0, 1, T)            # same shocks for every phi
fig, ax = plt.subplots(figsize=(11, 5))
for phi in [0.5, 0.9, 1.0]:
    y = np.zeros(T)
    for t in range(1, T):
        y[t] = phi * y[t-1] + eps[t]
    ax.plot(y, label=f'phi={phi}  (root z=1/phi={1/phi:.2f})')
ax.axhline(0, color='gray', lw=0.6)
ax.set_title('phi<1 mean-reverts; phi=1 is a random walk that wanders')
ax.set_xlabel('t'); ax.legend(); plt.tight_layout(); plt.show()

## 2. The regression the ADF test fits

Subtract $y_{t-1}$ from the AR(1) equation to turn the unit-root question into a
test on a single coefficient $\gamma = \phi - 1$:

$$\Delta y_t = \alpha + \beta t + \gamma\, y_{t-1} + \sum_{j=1}^{k}\delta_j\,\Delta y_{t-j} + \varepsilon_t$$

- $H_0:\ \gamma = 0$  (unit root, non-stationary)
- $H_1:\ \gamma < 0$  (stationary)

The lagged differences are the **augmentation** that whitens the residuals. Here
is a minimal hand-rolled DF regression (k=0) compared against statsmodels.

In [ ]:
def df_tstat(y):
    """Plain Dickey-Fuller t-stat on gamma via OLS of dy on [1, y_{t-1}]."""
    y = np.asarray(y, float)
    dy = np.diff(y)
    X = np.column_stack([np.ones(len(dy)), y[:-1]])   # [const, y_{t-1}]
    beta, *_ = np.linalg.lstsq(X, dy, rcond=None)
    resid = dy - X @ beta
    sigma2 = resid @ resid / (len(dy) - X.shape[1])
    cov = sigma2 * np.linalg.inv(X.T @ X)
    gamma, se = beta[1], np.sqrt(cov[1, 1])
    return gamma / se, gamma

walk = np.cumsum(rng.normal(0, 1, 300))               # random walk: unit root
stat, gamma = df_tstat(walk)
print(f'hand-rolled DF t-stat = {stat:.3f}  (gamma={gamma:+.4f})')
print(f'statsmodels ADF stat  = {adfuller(walk, maxlag=0, regression="c")[0]:.3f}')

### Validate: the DF t-statistic separates a random walk from a stationary series

A random walk has a unit root, so its Dickey-Fuller $t$-statistic sits **above** the 5%
critical value ($\approx -2.86$) — we fail to reject non-stationarity. A mean-reverting
(stationary) AR series gives a **strongly negative** $t$-statistic. We confirm both.

In [ ]:
t_walk, _ = df_tstat(walk)
ar = np.zeros(300)
r_ar = np.random.default_rng(1)
for t in range(1, 300):
    ar[t] = 0.5 * ar[t-1] + r_ar.normal()
t_ar, _ = df_tstat(ar)
print(f'random walk DF t = {t_walk:.2f}  (above -2.86 -> unit root NOT rejected)')
print(f'AR(0.5)     DF t = {t_ar:.2f}  (well below -2.86 -> stationary)')
assert t_walk > -2.86, 'a random walk fails to reject the unit root (non-stationary)'
assert t_ar < -2.86, 'a mean-reverting series gives a strongly negative DF t-stat (stationary)'
print('\n✅ the DF t-stat measures mean reversion: very negative = stationary')

## 3. Reading the statistic and the p-value

Under $H_0$ the statistic follows the **non-standard Dickey-Fuller distribution**,
so critical values are more negative than $-1.96$. Reject the unit root when the
statistic is **more negative** than the critical value (equivalently p < 0.05).

In [ ]:
def report_adf(series, regression='c', label=''):
    stat, pval, lag, nobs, crit, _ = adfuller(series, regression=regression, autolag='AIC')
    verdict = 'STATIONARY (reject H0)' if pval < 0.05 else 'NON-STATIONARY (fail to reject)'
    print(f'[{label:>9}] ADF={stat:7.3f}  p={pval:.4f}  lags={lag}  5%cv={crit["5%"]:.3f}  -> {verdict}')

trend = 50 + np.cumsum(2.0 + rng.normal(0, 3, 120))   # random walk WITH drift
report_adf(trend,          regression='ct', label='levels')   # trend -> use 'ct'
report_adf(np.diff(trend), regression='c',  label='diff')     # differenced -> 'c'

## 4. Cross-check with KPSS (null = stationary)

The ADF test has low power near $\phi \approx 1$, so pair it with KPSS, whose
null is the opposite. Agreement is a strong signal; disagreement flags trouble.

In [ ]:
import warnings; warnings.filterwarnings('ignore')   # KPSS p-value interpolation warnings
d = np.diff(trend)
adf_p  = adfuller(d, regression='c', autolag='AIC')[1]
kpss_p = kpss(d, regression='c', nlags='auto')[1]
print(f'Delta y:  ADF p={adf_p:.4f} (small=stationary)   KPSS p={kpss_p:.4f} (large=stationary)')
print('Both agree the differenced series is stationary.' if adf_p < 0.05 and kpss_p > 0.05
      else 'Tests disagree -> inspect for trend / structural break.')

### Validate: ADF and KPSS agree (complementary nulls)

ADF's null is "unit root" (non-stationary); KPSS's null is "stationary" — opposite hypotheses.
On a correctly differenced series they should **agree**: ADF rejects (small p) and KPSS fails to
reject (large p). Using both guards against each test's weaknesses. We confirm.

In [ ]:
print(f'differenced series: ADF p={adf_p:.4f} (small=stationary), KPSS p={kpss_p:.4f} (large=stationary)')
assert adf_p < 0.05, 'ADF rejects the unit root on the differenced series (stationary)'
assert kpss_p > 0.05, 'KPSS fails to reject stationarity -> both tests agree'
print('\n✅ confirm stationarity with BOTH tests — their opposite nulls catch each other’s blind spots')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **wrong deterministic term** | 'c' vs 'ct' flips the verdict on trending data (demo) |
| **ADF alone** | low power near a unit root; confirm with KPSS (verified) |
| **lag length** | too few lags → size distortion; use AIC-chosen lags |
| **structural breaks** | a break looks like a unit root; test for breaks separately |
| **short series** | unreliable; unit-root tests need reasonable sample sizes |

Demo: the deterministic term ('c' vs 'ct') flips the ADF verdict.

In [ ]:
# The subtle ADF gotcha: you must include the right DETERMINISTIC term. A series that is
# stationary around a linear TREND looks non-stationary if you run ADF with only a constant
# ('c'); adding the trend term ('ct') reveals the stationarity. The verdict flips on the
# regression term alone.
r_tr = np.random.default_rng(5)
trend_stat = 50 + 2 * np.arange(200) + r_tr.normal(0, 3, 200)   # stationary around a linear trend
p_c = adfuller(trend_stat, regression='c', autolag='AIC')[1]
p_ct = adfuller(trend_stat, regression='ct', autolag='AIC')[1]
print(f"ADF with 'c'  (constant only) : p={p_c:.4f}  -> looks NON-stationary")
print(f"ADF with 'ct' (const + trend) : p={p_ct:.4f}  -> correctly stationary")
assert p_c > 0.05 and p_ct < 0.05, 'the deterministic term flips the verdict — match it to the data'
print('\nChoose the regression term to match the data (trend -> ct), or ADF gives the wrong answer.')

## ✏️ Your turn

Write `integration_order(series)` that differences the series until the ADF test
rejects (p < 0.05) and returns the number of differences needed — that count is
the ARIMA $d$. Verify it returns `d = 1` for the drifting `trend` series above.

In [ ]:
def integration_order(series, max_d=2, alpha=0.05):
    s = np.asarray(series, float)
    # TODO(you): loop d = 0..max_d, run adfuller(s), return d when p < alpha,
    #            otherwise difference (s = np.diff(s)) and continue
    return None

d = integration_order(trend)
assert d == 1, f'expected d=1 for a drifting random walk, got {d}'
print(f'Chosen differencing order: d = {d}')

<details>
<summary>Solution</summary>

```python
def integration_order(series, max_d=2, alpha=0.05):
    s = np.asarray(series, float)
    for d in range(max_d + 1):
        if adfuller(s, autolag='AIC')[1] < alpha:
            return d
        s = np.diff(s)
    return max_d
```

</details>

## Key takeaways

- **ADF tests for a unit root:** a very negative $t$-statistic rejects non-stationarity
  (verified).
- **Confirm with KPSS:** its null is the opposite, so agreement is strong evidence (verified).
- **Pick the deterministic term:** 'c' vs 'ct' can flip the verdict on trending data (demo).
- **Difference, then re-test:** the differenced series should pass both tests before you model
  it.